In [ ]:
import glob

# fetch text files
train_txt_files = glob.glob("../input/tamil-wikipedia-articles/train/train/*.txt")
valid_txt_files = glob.glob("../input/tamil-wikipedia-articles/valid/valid/*.txt")

In [ ]:
ta_txt_files = train_txt_files + valid_txt_files

In [ ]:
print("Number of tamil text docs : ", len(ta_txt_files))

In [ ]:
ta_text = list()

for file in ta_txt_files:
    with open(file, 'r') as f:
        txt = ""
        word_count = 0
        for line in f:
            if word_count >= 448:
                word_count = 0
                if txt != "":
                    ta_text.append(txt)
                    txt = ""
                continue

            if line.startswith('<doc id='):
                if txt != "":
                    ta_text.append(txt)
                    txt = ""
                continue
                
            if not line.strip():
                continue
                
            d = line.strip()
            word_count += len(d.split())
            txt += d.strip("</doc>")

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', -1)

In [ ]:
ta_text_df = pd.DataFrame({"Text": ta_text})

In [ ]:
display(ta_text_df.head())

In [ ]:
import matplotlib.pyplot as plt

lengths = ta_text_df["Text"].apply(lambda x: len(x.split()))
plt.figure(figsize=(8, 8))
lengths.plot(kind="box")
plt.ylabel("Number of words")
plt.show()

print("Number of docs after chunking : ", ta_text_df.shape[0])
print("Number of docs with more than 448 tokens : ", sum(ta_text_df["Text"].apply(lambda x: len(x.split()) > 448)))
print("Maximum number of words in single chunk : ", max(lengths))
print("Mean Length: ", lengths.mean())

In [ ]:
def truncate(x, n_tokens = 448):
    tokens = x.split()
    if len(tokens) > n_tokens:
        return " ".join(tokens[:n_tokens])
    return x

In [ ]:
ta_text_df["Text"] = ta_text_df["Text"].apply(truncate)

In [ ]:
lengths = ta_text_df["Text"].apply(lambda x: len(x.split()))
plt.figure(figsize=(8, 8))
lengths.plot(kind="box")
plt.ylabel("Number of words")
plt.show()

print("Number of docs after chunking : ", ta_text_df.shape[0])
print("Number of docs with more than 448 tokens : ", sum(ta_text_df["Text"].apply(lambda x: len(x.split()) > 448)))
print("Maximum number of words in single chunk : ", max(lengths))
print("Mean Length: ", lengths.mean())

In [ ]:
ta_text_df.to_csv("Tamil wiki-data.csv", index = False)

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
stop_words = list()
with open("../input/tamil-stopwords/TamilStopWords.txt", 'r') as f:
    stop_words += f.readlines()
    
for i in range(len(stop_words)):
    stop_words[i] = stop_words[i].rstrip("\n")

In [ ]:
print(stop_words)

In [ ]:
def frequency_table(text) -> dict:
    words = word_tokenize(text)
    ps = PorterStemmer()
    
    freq_table = dict()
    for word in words:
        word = ps.stem(word)
        if word in stop_words:
            continue
        elif word in freq_table:
            freq_table[word] += 1
        else:
            freq_table[word] = 1
    return freq_table

In [ ]:
def score_sentences(tokenized_sentences, freq_table) -> dict:
    sentence_scores = dict()
    for sent in tokenized_sentences:
        word_count_in_sentence = (len(word_tokenize(sent)))
        word_count_in_sentence_wo_sw = 0    # Without stopwords
        for word in freq_table:
            if word in sent.lower():
                word_count_in_sentence_wo_sw += 1
                if sent in sentence_scores:
                    sentence_scores[sent] += freq_table[word]
                else:
                    sentence_scores[sent] = freq_table[word]
        if sent in sentence_scores:
            sentence_scores[sent] = sentence_scores[sent] / word_count_in_sentence_wo_sw
    return sentence_scores

In [ ]:
def avg(sentence_scores) -> float:
    average = 0
    for key in sentence_scores:
        average += sentence_scores[key]
    # len(sentence_scores) becomes zero when stopwords 
    # are removed and no word left in the doc
    average = average / (len(sentence_scores) + 1e-100) # Avoid zero div error
    return average 

In [ ]:
def summarize(tokenized_sentences, sentence_scores, threshold) -> str:
    sentence_count = 0
    summary = ""
    for sent in tokenized_sentences:
        if (sent in sentence_scores 
            and sentence_scores[sent] >= threshold):
            summary += " " + sent
            sentence_count += 1
    return summary

In [ ]:
def summarize_series(text_series) -> pd.DataFrame:
    print(f"Total number of docs: {text_series.shape[0]}")
    summarized_text = list()
    for i in range(text_series.shape[0]):
        freq_table = frequency_table(text_series[i])
        tokenized_sentences = sent_tokenize(text_series[i])
        sentence_scores = score_sentences(tokenized_sentences, freq_table)
        threshold = avg(sentence_scores)
        summary = summarize(tokenized_sentences, sentence_scores, threshold)
        summarized_text.append(summary)
        print(f"Number of docs summarized: {i}", end='\r')
    print("<"+ "=" * 40 + " Completed summarizing " + "=" * 40 + ">")
    return pd.DataFrame({"Original": text_series, "Summary": summarized_text})

In [ ]:
ta_text_summarized_df = summarize_series(ta_text_df["Text"])

In [ ]:
for i in range(5):
    print("Original : ", ta_text_summarized_df["Original"][i])
    print()
    print("Summary : ", ta_text_summarized_df["Summary"][i])
    print()

In [ ]:
ta_text_summarized_df.to_csv("Tamil wiki-data summarized.csv", index = False)

In [ ]:
plt.figure(figsize=(8, 8))
lengths = ta_text_summarized_df["Summary"].apply(lambda x: len(x.split()))
lengths.plot(kind="box")
plt.ylabel("Number of words")
plt.show()

print("Mean Length after summarization : ", lengths.mean())